# training Tversky similarity layer in triplet loss

In [1]:
from utils import *
from tversky_sirl import *

FPE_SPLIT_SEED = 42
SIRL_SEED = 0
n_queries = 1000

print(f"\n=== TRAINING SIRL FOR {n_queries} QUERIES ===")
anchors, positives, negatives = load_data(n_queries)

print(f"  -- seed {SIRL_SEED} --")
set_all_seeds(SIRL_SEED)

model, _ = train_tversky_sirl(
    anchors, positives, negatives,
    fbank_size=4,
    use_symmetric_loss=True,
)

pybullet build time: Oct 21 2025 16:30:45



=== TRAINING SIRL FOR 1000 QUERIES ===
  -- seed 0 --
Epoch    0 | loss=1.9955 | triplet_acc=0.504 | lr=0.00400
Epoch  100 | loss=2.0000 | triplet_acc=0.000 | lr=0.00400
Epoch  200 | loss=2.0000 | triplet_acc=0.000 | lr=0.00399
Epoch  300 | loss=2.0000 | triplet_acc=0.000 | lr=0.00399
Epoch  400 | loss=2.0000 | triplet_acc=0.000 | lr=0.00398
Epoch  500 | loss=2.0000 | triplet_acc=0.000 | lr=0.00398
Epoch  600 | loss=2.0000 | triplet_acc=0.000 | lr=0.00398
Epoch  700 | loss=2.0000 | triplet_acc=0.000 | lr=0.00397
Epoch  800 | loss=2.0000 | triplet_acc=0.000 | lr=0.00397
Epoch  900 | loss=2.0000 | triplet_acc=0.000 | lr=0.00396
Epoch 1000 | loss=2.0000 | triplet_acc=0.000 | lr=0.00396
Epoch 1100 | loss=2.0000 | triplet_acc=0.000 | lr=0.00396
Epoch 1200 | loss=2.0000 | triplet_acc=0.000 | lr=0.00395
Epoch 1300 | loss=2.0000 | triplet_acc=0.000 | lr=0.00395
Epoch 1400 | loss=2.0000 | triplet_acc=0.000 | lr=0.00394
Epoch 1500 | loss=2.0000 | triplet_acc=0.000 | lr=0.00394
Epoch 1600 | loss

In [2]:
OUT_DIR = "tversky_feats_results"
model.save_model(f"{OUT_DIR}/tversky_4.pth")

model saved to tversky_feats_results/tversky_4.pth


# feature analysis

for each of table, laptop:
    find trajectory with max(feature value)
    find trajectory with min(feature value)
    do max - min, retrieve set of trajectories
    visualize (sample)
    measure average feature value in retrieved set, compare to average of all trajectories (or average of all not in set)

In [99]:
# CHECKPOINT!
from utils import *
from tversky_sirl import *
OUT_DIR = "tversky_feats_results"
model = load_model(f"{OUT_DIR}/tversky_4.pth")


In [100]:
model

TverskySIRL(
  (encoder): Sequential(
    (0): Linear(in_features=567, out_features=1024, bias=True)
    (1): ReLU()
    (2): Linear(in_features=1024, out_features=1024, bias=True)
    (3): ReLU()
    (4): Linear(in_features=1024, out_features=6, bias=True)
  )
  (tversky_sim): TverskySimilarity(
    (feature_bank): Embedding(4, 6)
  )
)

In [101]:
all_trajs = load_all_trajs()
all_features = get_features(all_trajs, all_trajs)

all_trajs shape: (10000, 21, 97)
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis elem

In [102]:
print(f"table min value is {np.min(all_features[:,0])} at idx {np.argmin(all_features[:,0])}")
print(f"table max value is {np.max(all_features[:,0])} at idx {np.argmax(all_features[:,0])}")
print(f"laptop min value is {np.min(all_features[:,1])} at idx {np.argmin(all_features[:,1])}")
print(f"table max value is {np.max(all_features[:,1])} at idx {np.argmax(all_features[:,1])}")

table min value is 0.0 at idx 6000
table max value is 1.0 at idx 3421
laptop min value is 0.0 at idx 4894
table max value is 1.0 at idx 5018


    do max - min, retrieve set of trajectories
    visualize (sample)
    measure average feature value in retrieved set, compare to average of all trajectories (or average of all not in set)

In [103]:
# from tversky-networks-iclr2026/experiments/103-vision-nabirds/src/semantic_utils.py
def retrieve_semantic_expression(
    instance_vectors: torch.Tensor,   # (N, D)  — all instance reps from parquet
    feature_bank: torch.Tensor,        # (F, D)  — from .npy
    expression: str,
    top_feature_count: int,
    top_result_count: int,
) -> dict:
    """
    Evaluate a set expression over instance vectors and feature bank.
    Expression uses s(i) notation where i is a dataset item_id (row index).
    Example: "s(0) - s(1)"  →  features of item 0 minus features of item 1
    """
    query_item_ixes = []

    def s(item_ix: int) -> set:
        print(item_ix)
        feature_values = instance_vectors[item_ix:item_ix+1] @ feature_bank.T  # (1, F)
        print("feature_values")
        print(feature_values)
        feature_ixes = []
        for feature_ix in torch.argsort(feature_values[0], descending=True)[:top_feature_count]:
            print("loop")
            print(feature_values[0][feature_ix])
            if feature_values[0][feature_ix] > 0:
                feature_ixes.append(int(feature_ix))
            else:
                break
        query_item_ixes.append(item_ix)
        print(set(feature_ixes))
        return set(feature_ixes)

    semantic_features = eval(expression)

    if not semantic_features:
        # logging.warning("Expression evaluated to empty feature set.")
        return {"expression": expression, "query_item_ixes": [], "top_instances": [], "feature_count": 0}

    semantic_f_bank = torch.index_select(
        feature_bank, 0, torch.tensor(sorted(semantic_features))
    )
    dot = instance_vectors @ semantic_f_bank.T          # (N, |features|)
    p_saliences = F.relu(dot).sum(dim=1)                # (N,)
    p_measures  = dot.sum(dim=1)                        # (N,)

    top_instances = []
    for result_ix in torch.argsort(p_measures, descending=True)[:top_result_count]:
        top_instances.append({
            "item_ix"  : result_ix.item(),
            "salience" : p_saliences[result_ix].item(),
            "measure"  : p_measures[result_ix].item(),
        })

    return {
        "expression"      : expression.strip(),
        "query_item_ixes" : query_item_ixes,
        "feature_count"   : len(semantic_features),
        "top_instances"   : top_instances,
    }

In [104]:
feature_bank = model.tversky_sim.feature_bank.weight.detach()
feature_bank.shape # 4 Tversky features, of 6-dim representation vectors

torch.Size([4, 6])

In [105]:
table_min_traj = all_trajs[np.argmin(all_features[:,0])]
table_max_traj = all_trajs[np.argmax(all_features[:,0])]
laptop_min_traj = all_trajs[np.argmin(all_features[:,1])]
laptop_max_traj = all_trajs[np.argmax(all_features[:,1])]

In [106]:
normalized_trajs = tx(np.array([
    table_min_traj,
    table_max_traj,
    laptop_min_traj,
    laptop_max_traj
]))
normalized_trajs.shape

(4, 21, 27)

In [107]:
embeds = model(torch.tensor(trajs)).detach()
embeds.shape

torch.Size([4, 6])

In [108]:
retrieve_semantic_expression(
    instance_vectors=embeds, #(N, D)
    feature_bank=feature_bank, # (F, D)
    expression="s(1)-s(0)", # table_max - table_min
    top_feature_count=4,
    top_result_count=10,
)

1
feature_values
tensor([[-650.0502, -980.3074, -722.1364, -633.4221]])
loop
tensor(-633.4221)
set()
0
feature_values
tensor([[-496.7783, -749.1676, -551.7877, -484.0378]])
loop
tensor(-484.0378)
set()


{'expression': 's(1)-s(0)',
 'query_item_ixes': [],
 'top_instances': [],
 'feature_count': 0}

all the feature values for these trajectories are negative?? let's see if there are any in the set of all trajectories with high salience.

In [109]:
normalized_all_trajs = tx(all_trajs)
all_embeds = model(torch.tensor(normalized_all_trajs)).detach()
all_embeds.shape # (N, D)

torch.Size([10000, 6])

In [110]:
# from tversky/test_mnist.py
# edited
import torch.nn.functional as F
def compute_salience(x, feature_bank):
    """
    x is (N, D)
    feature_bank is (F, D)
    """
    feature_measures = x @ feature_bank.T # (N, F)
    salience_measures = F.relu(feature_measures).sum(-1)
    return salience_measures

In [111]:
max(compute_salience(all_embeds, feature_bank))

tensor(0.)

interesting, so looks like all the computed features are negative... I wonder if this is because I negated the similarity to get distance (in `tversky_sirl:46`)?

# crazy shit o'clock

In [156]:
all_inv_salience = compute_salience(all_embeds, -feature_bank) # NOTE NEGATED feature bank to make all features positive
min_salience_traj = all_trajs[np.argmin(all_inv_salience).item()]
max_salience_traj = all_trajs[np.argmax(all_inv_salience).item()]

In [113]:
print(f"min salience is {min(all_inv_salience)}")
print(f"max salience is {max(all_inv_salience)}")

min salience is 1955.9285888671875
max salience is 3132.606689453125


In [153]:
import sys
sys.path.insert(0, '..')
import numpy as np
import plotly.graph_objects as go
from src.envs.jacorobot import Jacorobot
from src.utils.bullet_utils import waypts_to_xyz

def vis_trajectory(traj, title=""):
    env_params = {
        "object_centers": {'HUMAN_CENTER': [-0.2, -0.6, 0.9], 'LAPTOP_CENTER': [-0.5, 0.0, 0.635]},
        "resources_dir": "../data/resources",
        "horizon": 10,
        "timestep": 0.5,
        "real": False,
    }

    env = Jacorobot(
        env_params["object_centers"],
        env_params["resources_dir"],
        env_params["horizon"],
        env_params["timestep"],
        debug=False  # headless — no GUI needed
    )

    xyz = waypts_to_xyz(env.objectID["robot"], traj)

    fig = go.Figure()
    fig.add_trace(go.Scatter3d(
        x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
        mode='lines+markers'
    ))

    posH = env_params["object_centers"]["HUMAN_CENTER"]
    posL = env_params["object_centers"]["LAPTOP_CENTER"]

    fig.add_trace(go.Scatter3d(
        x=[posH[0], posL[0]],
        y=[posH[1], posL[1]],
        z=[posH[2], posL[2]],
        mode='markers',
        marker=dict(
            color=["gray", "black"],
            symbol=["cross", "square"],
            size=8,
            showscale=False
        ),
        name="human (+), laptop (square)"
    ))
    fig.update_layout(title=title)

    config = {
    'toImageButtonOptions': {
        'format': 'png', # or jpeg, svg, webp
        'scale': 4 # Increase this for higher resolution
    }
    }
    fig.show(config=config)


In [157]:
vis_trajectory(max_salience_traj, "max salience traj")

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0

In [158]:
vis_trajectory(min_salience_traj, "min salience traj")

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0

# semantic expressions with inverted feature values

In [94]:
# from tversky-networks-iclr2026/experiments/103-vision-nabirds/src/semantic_utils.py
def retrieve_semantic_expression(
    instance_vectors: torch.Tensor,   # (N, D)  — all instance reps from parquet
    feature_bank: torch.Tensor,        # (F, D)  — from .npy
    expression: str,
    top_feature_count: int,
    top_result_count: int,
) -> dict:
    """
    Evaluate a set expression over instance vectors and feature bank.
    Expression uses s(i) notation where i is a dataset item_id (row index).
    Example: "s(0) - s(1)"  →  features of item 0 minus features of item 1
    """
    query_item_ixes = []

    def s(item_ix: int) -> set:
        print(item_ix)
        feature_values = instance_vectors[item_ix:item_ix+1] @ feature_bank.T  # (1, F)
        print("feature_values")
        print(feature_values)
        feature_ixes = []
        for feature_ix in torch.argsort(feature_values[0], descending=True)[:top_feature_count]:
            print("loop")
            print(feature_values[0][feature_ix])
            if feature_values[0][feature_ix] > 0:
                feature_ixes.append(int(feature_ix))
            else:
                break
        query_item_ixes.append(item_ix)
        print("ret")
        print(set(feature_ixes))
        return set(feature_ixes)

    semantic_features = eval(expression)

    if not semantic_features:
        print("Expression evaluated to empty feature set.")
        return {"expression": expression, "query_item_ixes": [], "top_instances": [], "feature_count": 0}

    semantic_f_bank = torch.index_select(
        feature_bank, 0, torch.tensor(sorted(semantic_features))
    )
    dot = instance_vectors @ semantic_f_bank.T          # (N, |features|)
    p_saliences = F.relu(dot).sum(dim=1)                # (N,)
    p_measures  = dot.sum(dim=1)                        # (N,)

    top_instances = []
    for result_ix in torch.argsort(p_measures, descending=True)[:top_result_count]:
        top_instances.append({
            "item_ix"  : result_ix.item(),
            "salience" : p_saliences[result_ix].item(),
            "measure"  : p_measures[result_ix].item(),
        })

    return {
        "expression"      : expression.strip(),
        "query_item_ixes" : query_item_ixes,
        "feature_count"   : len(semantic_features),
        "top_instances"   : top_instances,
    }

In [ ]:
retrieve_semantic_expression(
    instance_vectors=embeds, #(N, D)
    feature_bank=-feature_bank, # (F, D)
    expression="s(1)-s(0)", # table_max - table_min
    top_feature_count=1,
    top_result_count=10,
)

3
feature_values
tensor([[584.6979, 881.7628, 649.4648, 569.7318]])
loop
tensor(881.7628)
ret
{1}
2
feature_values
tensor([[511.6690, 771.6029, 568.4320, 498.5082]])
loop
tensor(771.6029)
ret
{1}
Expression evaluated to empty feature set.


{'expression': 's(3)-s(2)',
 'query_item_ixes': [],
 'top_instances': [],
 'feature_count': 0}

all of these trajectories express all 4 features, and in the same order, such that the set difference between sets of features is always the empty set.

# centering embeddings

In [122]:
all_embeds

tensor([[-201.4079, -259.0656, -226.8636, -264.1326, -221.1614, -210.5917],
        [-224.3506, -288.5919, -252.7414, -294.2239, -246.3143, -234.4906],
        [-208.1220, -267.7359, -234.4510, -272.9745, -228.5107, -217.5762],
        ...,
        [-211.1855, -271.7306, -237.9076, -277.1583, -231.9547, -220.9170],
        [-217.9283, -280.4059, -245.5719, -286.0105, -239.3212, -227.8959],
        [-229.3984, -295.1524, -258.4758, -301.0257, -251.9179, -239.8826]])

In [123]:
feature_bank

tensor([[3.4092e-01, 1.0689e-01, 4.5169e-01, 7.9888e-01, 5.5463e-01, 3.8456e-02],
        [5.6685e-05, 5.4320e-01, 9.3193e-01, 8.9804e-01, 7.7458e-01, 2.5921e-01],
        [2.9108e-01, 7.9080e-01, 2.1747e-02, 6.5505e-01, 3.0114e-01, 4.3902e-01],
        [3.5390e-01, 6.5219e-01, 4.7014e-01, 4.2593e-01, 1.8187e-01, 1.2892e-01]])

actually, turns out the saliences are all negative because the embeddings are negative, not the feature bank.
what happens if we try centering them (subtracting their mean)?

In [125]:
mu = all_embeds.mean(0)        # shape [6]
centered_embeds = all_embeds - mu     # [10000, 6]

In [126]:
centered_embeds

tensor([[20.9173, 26.9863, 23.6061, 27.5653, 22.9839, 21.8607],
        [-2.0254, -2.5399, -2.2717, -2.5260, -2.1690, -2.0382],
        [14.2032, 18.3160, 16.0186, 18.7233, 15.6346, 14.8762],
        ...,
        [11.1397, 14.3213, 12.5620, 14.5396, 12.1906, 11.5355],
        [ 4.3969,  5.6461,  4.8978,  5.6874,  4.8241,  4.5565],
        [-7.0732, -9.1005, -8.0061, -9.3278, -7.7726, -7.4302]])

In [130]:
all_salience = compute_salience(centered_embeds, feature_bank) 
min_salience_idx = np.argmin(all_salience).item()
min_salience_traj = all_trajs[min_salience_idx]
max_salience_idx = np.argmax(all_salience).item()
max_salience_traj = all_trajs[max_salience_idx]

In [134]:
print(f"min salience traj has features {all_features[min_salience_idx]}")
print(f"max salience traj has features {all_features[max_salience_idx]}")

min salience traj has features [0.26290355 0.68076226]
max salience traj has features [0.21662685 0.33329545]


In [128]:
vis_trajectory(min_salience_traj, "min salience traj")

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0

In [129]:
vis_trajectory(max_salience_traj, "max salience traj")

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0

# visualizing features

In [136]:
tversky_feature_measures = centered_embeds @ feature_bank.T # (N, F)
tversky_feature_measures.shape

torch.Size([10000, 4])

In [147]:
tversky_feature_measures

tensor([[ 56.2877,  84.8837,  62.5182,  54.8400],
        [ -5.2874,  -7.9737,  -5.8502,  -5.1745],
        [ 38.2365,  57.6590,  42.4709,  37.2390],
        ...,
        [ 29.8229,  44.9769,  33.1006,  29.0855],
        [ 11.7091,  17.6568,  13.0299,  11.4282],
        [-19.0489, -28.7283, -21.1425, -18.5469]])

In [146]:
for tversky_feature_idx in range(4):
    min_idx = np.argmin(tversky_feature_measures[:,tversky_feature_idx].detach()).item()
    min_traj = all_trajs[min_idx]
    max_idx = np.argmax(tversky_feature_measures[:,tversky_feature_idx].detach()).item()
    max_traj = all_trajs[max_idx]
    vis_trajectory(min_traj, f"min value of tversky feature {tversky_feature_idx}")
    vis_trajectory(max_traj, f"max value of tversky feature {tversky_feature_idx}")

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:


urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_shoulder
b3Warning[examples/Importers/ImportURDFDemo/B

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0

these all recover the same max salience, min salience examples... that means that there is a global max (highest on all feature values) and a global min (lowest on all feature values).

# correlating tversky features with ground truth features

In [152]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# to numpy
tv = tversky_feature_measures.detach().cpu().numpy()   # (10000, 4)
gt = all_features

gt_names = ["table", "laptop"]
tv_names = [f"tversky_{k}" for k in range(tv.shape[1])]

# rows = ground truth (2), cols = tversky feature (4)
fig = make_subplots(
    rows=len(gt_names), cols=tv.shape[1],
    shared_xaxes=False, shared_yaxes=False,
    horizontal_spacing=0.05, vertical_spacing=0.12,
    subplot_titles=[
        f"{g} vs {t} (r={np.corrcoef(gt[:, gi], tv[:, ti])[0,1]:.2f})"
        for gi, g in enumerate(gt_names) for ti, t in enumerate(tv_names)
    ],
)

for gi in range(len(gt_names)):
    for ti in range(tv.shape[1]):
        fig.add_trace(
            go.Scattergl(
                x=gt[:, gi], y=tv[:, ti],
                mode="markers",
                marker=dict(size=3, opacity=0.35),
                showlegend=False,
            ),
            row=gi + 1, col=ti + 1,
        )
        fig.update_xaxes(title_text=gt_names[gi], row=gi + 1, col=ti + 1)
        fig.update_yaxes(title_text=tv_names[ti], row=gi + 1, col=ti + 1)

fig.update_layout(height=600, width=1300, title="Tversky feature measures vs ground-truth features")
config = {
  'toImageButtonOptions': {
    'format': 'png', # or jpeg, svg, webp
    'scale': 4 # Increase this for higher resolution
  }
}
fig.show(config=config)

Tversky dimensions are collapsed... learning the same info at 4 different scales!

In [173]:
import torch

X = all_embeds - all_embeds.mean(0, keepdim=True)   # center! rank of variation, not of the mean offset
s = torch.linalg.svdvals(X)                          # singular values, descending, shape [6]

# --- participation ratio (variance-based effective rank) ---
var = s**2                                           # eigenvalues of the covariance (∝)
pr = var.sum()**2 / (var**2).sum()                   # = (Σλ)² / Σλ²

# --- spectral entropy effective rank (Roy & Vetterli) ---
p = var / var.sum()
H = -(p * (p + 1e-12).log()).sum()
erank = H.exp()

print("singular values:", s.tolist())
print("variance ratios :", (var / var.sum()).cumsum(0).tolist())  # cumulative explained variance
print(f"participation ratio: {pr:.2f}")
print(f"entropy erank:       {erank:.2f}")

singular values: [4075.019775390625, 6.689264297485352, 3.662869691848755, 2.2708215713500977, 0.8830038905143738, 0.477588415145874]
variance ratios : [0.9999961256980896, 0.9999988079071045, 0.9999996423721313, 0.9999999403953552, 1.0, 1.0]
participation ratio: 1.00
entropy erank:       1.00


... because the embeddings are degenerate

In [174]:
print("mean norm:", all_embeds.mean(0).norm().item())
print("centered first-col mean:", (all_embeds - all_embeds.mean(0,keepdim=True)).mean(0).abs().max().item())  # should be ~0

mean norm: 626.672119140625
centered first-col mean: 2.0382691218401305e-05


In [175]:
import numpy as np
print("corr(table, laptop):", np.corrcoef(gt[:,0], gt[:,1])[0,1])

corr(table, laptop): 0.041106979500096646


... the features are independent though... maybe the trajectories in our sample don't vary enough across one or both of the feature dims?

In [178]:
import numpy as np
DATA_DIR = "../data/Data Uploads"
def load_data(n_queries, data_dir=DATA_DIR, per_file=100, start_from_idx = 0):
    """
    loads query data, pass in number of queries from 100-1000
    """
    assert n_queries % per_file == 0, f"n_queries must be a multiple of {per_file}"

    n_files = n_queries // per_file

    all_anchors, all_pos, all_neg = [], [], []
    for i in range(n_files):
        path = f"{data_dir}/sim_{per_file}_{i + start_from_idx}.npz"
        with np.load(path) as data:
            all_anchors.append(data['anchors'])
            all_pos.append(data['positives'])
            all_neg.append(data['negatives'])

    # concat queries from all files
    anchors, positives, negatives = (np.concatenate(all_anchors, axis=0), np.concatenate(all_pos, axis=0), np.concatenate(all_neg, axis=0))
    return anchors, positives, negatives

In [179]:
anchors, positives, negatives = load_data(1000)
all_human_trajs = np.concatenate([anchors, positives, negatives], axis=0)
all_human_trajs.shape

(3000, 21, 97)

In [182]:
all_human_features = get_features(all_trajs, all_human_trajs)

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0

In [183]:
print(f"table min value is {np.min(all_human_features[:,0])} at idx {np.argmin(all_human_features[:,0])}")
print(f"table max value is {np.max(all_human_features[:,0])} at idx {np.argmax(all_human_features[:,0])}")
print(f"laptop min value is {np.min(all_human_features[:,1])} at idx {np.argmin(all_human_features[:,1])}")
print(f"table max value is {np.max(all_human_features[:,1])} at idx {np.argmax(all_human_features[:,1])}")

table min value is 0.05046888071512884 at idx 411
table max value is 0.9868542954312279 at idx 684
laptop min value is 0.08139430914002162 at idx 2678
table max value is 0.9615804989879095 at idx 1436


no, the feature values have pretty good spread in the collected trajectories.